# Generate an earnings preview for the next scheduled event

Runs the earnings-preview skill (`src/preview_agent/workspace/.claude/skills/earnings-preview/SKILL.md`) through the Claude Agent SDK for one event from the competition calendar, the same way the competition produces its previews: a one-line request naming the company, ticker, fiscal period and scheduled time, with the agent researching the rest on the public web.

**Knowledge cutoff.** A preview may only use information public before the event's `knowledge_cutoff`, so only events whose cutoff is still ahead are eligible. The run refuses to start after the cutoff, stops when it arrives, and marks the preview `usable` only if it finished before.

**Cost.** `SETTINGS` picks a preset from `generator.PRESETS`:

- `"cheap"`: `claude-haiku-5-5`, low effort, at most 12 turns, 4 web searches and 3 page fetches (hard caps; extra calls are refused), stopped near $0.10. Searches cost $0.01 each.
- `"default"`: `claude-opus-5-5`, 10 turns, stopped near $0.05, no web caps.

Override single fields with `generator.resolve_settings("cheap", max_searches=6)`. The budget stop is only a safety net: a run stopped by it returns no preview. The run cell does nothing until you set `RUN = True`.

Output: `data/surprise/own_previews/<event_id>_<UTC start>.md` and a `.json` record of the run.

In [6]:
%load_ext autoreload
%autoreload 2

import json
from datetime import datetime, timezone

import pandas as pd
from IPython.display import Markdown, display

from preview_agent import events, generator

pd.set_option("display.width", 200)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Calendar

Scheduled events from now to 60 days ahead. `cutoff_ahead` marks the ones that can still be previewed.

In [7]:
calendar = events.fetch_events()
table = events.events_table(calendar)
print(f"{len(table)} scheduled events | cutoff still ahead: {int(table['cutoff_ahead'].sum())}")
table[table["cutoff_ahead"]].head(15)

3818 scheduled events | cutoff still ahead: 3796


,event_id,tickers,event_datetime,knowledge_cutoff,event_type,hours_to_cutoff,cutoff_ahead
22,ea_AEYE_Q3_2026,AEYE,2026-10-26 20:01:00+00:00,2026-10-26 20:00:00+00:00,EARNINGS_RELEASE,42.7,True
23,ea_DORM_Q3_2026,DORM,2026-10-26 20:01:00+00:00,2026-10-26 20:00:00+00:00,EARNINGS_RELEASE,42.7,True
24,ea_FSBC_Q3_2026,FSBC,2026-10-26 20:01:00+00:00,2026-10-26 20:00:00+00:00,EARNINGS_RELEASE,42.7,True
25,ea_PSMT_Q4_2026,PSMT,2026-10-26 20:01:00+00:00,2026-10-26 20:00:00+00:00,EARNINGS_RELEASE,42.7,True
26,ea_CDNS_Q3_2026,CDNS,2026-10-26 20:02:00+00:00,2026-10-26 20:00:00+00:00,EARNINGS_RELEASE,42.7,True
27,ea_AMKR_Q3_2026,AMKR,2026-10-26 20:03:00+00:00,2026-10-26 20:00:00+00:00,EARNINGS_RELEASE,42.7,True
28,ea_KFRC_Q3_2026,KFRC,2026-10-26 20:04:00+00:00,2026-10-26 20:00:00+00:00,EARNINGS_RELEASE,42.7,True
29,ea_AGYS_Q2_2027,AGYS,2026-10-26 20:05:00+00:00,2026-10-26 20:00:00+00:00,EARNINGS_RELEASE,42.7,True
30,ea_BRX_Q3_2026,BRX,2026-10-26 20:05:00+00:00,2026-10-26 20:00:00+00:00,EARNINGS_RELEASE,42.7,True
31,ea_CINF_Q3_2026,CINF,2026-10-26 20:05:00+00:00,2026-10-26 20:00:00+00:00,EARNINGS_RELEASE,42.7,True


## 2. Choose the event

By default, the next scheduled event whose cutoff has not passed. Set `TICKER` to pick the next event for one company instead. The calendar has no company names, so the request looks the name up from the ticker on Yahoo Finance (cached in `data/surprise/company_names.json`) and names the ticker only if that fails. Set `COMPANY` to override.

In [8]:
TICKER = None   # e.g. "VZ"
COMPANY = None  # None: look the name up from the ticker
SETTINGS = generator.resolve_settings("cheap")  # or "default", or overrides: resolve_settings("cheap", max_searches=6)

event = events.next_event(calendar, ticker=TICKER)
hours_left = (generator.parse_time(event["knowledge_cutoff"]) - datetime.now(timezone.utc)).total_seconds() / 3600
print(json.dumps(event, indent=1))
print(f"\nknowledge cutoff in {hours_left:.1f} hours")

{
 "event_id": "ea_AEYE_Q3_2026",
 "event_type": "EARNINGS_RELEASE",
 "timing_category": "SCHEDULED",
 "event_datetime": "2026-10-26T20:01:00Z",
 "knowledge_cutoff": "2026-10-26T20:00:00Z",
 "focal_assets": [
  {
   "identifier_type": "TICKER",
   "identifier_value": "AEYE"
  }
 ]
}

knowledge cutoff in 42.7 hours


## 3. Dry run: what will be sent

In [9]:
options = generator.build_options(SETTINGS)
print(SETTINGS)
print("prompt:", f"/{generator.SKILL_NAME} {generator.build_request(event, COMPANY)}")
print("skill: ", generator.SKILL_PATH.relative_to(generator.ROOT), "| sha256", generator.skill_sha256()[:12])
print({k: getattr(options, k) for k in ("model", "effort", "setting_sources", "skills", "allowed_tools", "disallowed_tools", "max_turns", "max_budget_usd")})
print("cost note:", generator.cost_note(SETTINGS))

PreviewSettings(model='claude-haiku-5-5', effort='low', max_turns=12, max_budget_usd=0.1, max_searches=4, max_fetches=3)
prompt: /earnings-preview AudioEye, Inc. (AEYE), fiscal period Q3 FY2026, earnings release scheduled 2026-10-26 20:01 UTC. Knowledge cutoff: 2026-10-26T20:00:00Z.
skill:  src/preview_agent/workspace/.claude/skills/earnings-preview/SKILL.md | sha256 a30d383a6f1c
{'model': 'claude-haiku-5-5', 'effort': 'low', 'setting_sources': ['project'], 'skills': ['earnings-preview'], 'allowed_tools': ['WebSearch', 'WebFetch', 'Read'], 'disallowed_tools': ['Bash', 'Write', 'Edit', 'Agent', 'Task'], 'max_turns': 12, 'max_budget_usd': 0.1}
cost note: Cost limit for this task: use at most 4 web searches and at most 3 page fetches, then write the preview from what you have. Calls beyond the limit are refused.


## 4. Run (paid)

Set `RUN = True` to generate the preview. Progress (skills loaded, each web search and fetch) prints as it goes.

In [11]:
RUN = True

if RUN:
    record = await generator.generate_preview(event, company=COMPANY, settings=SETTINGS)
else:
    record = None
    print("Set RUN = True to generate the preview.")

prompt: /earnings-preview AudioEye, Inc. (AEYE), fiscal period Q3 FY2026, earnings release scheduled 2026-10-26 20:01 UTC. Knowledge cutoff: 2026-10-26T20:00:00Z.
session started; skills loaded: ['earnings-preview', 'deep-research', 'design', 'design-sync', 'dataviz', 'update-config', 'verify', 'debug', 'code-review', 'simplify', 'batch', 'fewer-permission-prompts', 'doctor', 'loop', 'claude-api', 'workflow-authoring', 'run', 'run-skill-generator', 'plugin-authoring']
  ToolSearch: select:WebSearch,WebFetch
  Grep: 
  WebSearch: AudioEye AEYE Q3 2026 earnings consensus revenue EPS estimate
  WebSearch: AudioEye AEYE Q2 2026 results guidance earnings call
  WebSearch: AudioEye stock earnings reaction history
  (text, 5875 chars)
success | turns 6 | cost $0.041 | web {'WebSearch': 3} | usable True


In [12]:
if record:
    summary = {k: record.get(k) for k in ("event_id", "subtype", "usable", "finished_before_cutoff", "num_turns", "total_cost_usd", "web_calls", "skills_loaded", "started_at", "finished_at")}
    print(json.dumps(summary, indent=1, default=str))
    print(f"{len(record['tool_calls'])} tool calls | saved: {record['files']}")
    if record["preview"]:
        display(Markdown(record["preview"]))

{
 "event_id": "ea_AEYE_Q3_2026",
 "subtype": "success",
 "usable": true,
 "finished_before_cutoff": true,
 "num_turns": 6,
 "total_cost_usd": 0.041126275,
 "web_calls": {
  "WebSearch": 3
 },
 "skills_loaded": [
  "earnings-preview",
  "deep-research",
  "design",
  "design-sync",
  "dataviz",
  "update-config",
  "verify",
  "debug",
  "code-review",
  "simplify",
  "batch",
  "fewer-permission-prompts",
  "doctor",
  "loop",
  "claude-api",
  "workflow-authoring",
  "run",
  "run-skill-generator",
  "plugin-authoring"
 ],
 "started_at": "2026-10-25T01:15:53.515907+00:00",
 "finished_at": "2026-10-25T01:16:18.224824+00:00"
}
5 tool calls | saved: ['/Users/yuhaohuo/Desktop/code/explaning_market/data/surprise/own_previews/ea_AEYE_Q3_2026_20261025T011553Z.md', '/Users/yuhaohuo/Desktop/code/explaning_market/data/surprise/own_previews/ea_AEYE_Q3_2026_20261025T011553Z.json']


# AudioEye (AEYE) Q3 FY2026 Earnings Preview

**Company:** AudioEye, Inc. (NASDAQ: AEYE)
**Quarter:** Q3 FY2026 (quarter ended Sep 30, 2026)
**Earnings date:** You gave 2026-10-26 20:01 UTC. One aggregator shows Nov 3, 2026 instead. I could not confirm either date from a primary source, so verify on the IR page before trading around it.

**Data caveat:** Today is 2026-10-24, so the Oct 26 date is in the future and the knowledge cutoff you gave (2026-10-26T20:00Z) is also later than today. Figures below come from the Aug 13, 2026 Q2 release and third-party estimates, which may have moved since.

## 1. Consensus Estimates (as of late Oct 2026; sources disagree)

| Metric | Company guidance (Q3) | Consensus (varies by source) | Notes |
|---|---|---|---|
| Revenue | $10.85M – $11.05M (midpoint $10.95M) | ~$11.13M – $11.2M | Guidance midpoint sits **below** Street |
| Adj. EBITDA | $3.4M – $3.6M | Not found | |
| EPS | Not guided quarterly | $0.23 (MarketBeat) or $0.26 (Benzinga/StockAnalysis) | Source conflict |
| FY26 revenue | $43.5M – $44.0M (tightened) | ~$43.9M | In line with midpoint |
| FY26 adj. EPS | ≥ $0.98 | $0.85 – $0.94 | Guidance above Street range |

Sources: [MarketBeat Q3 guidance](https://www.marketbeat.com/instant-alerts/audioeye-nasdaqaeye-issues-q3-2026-earnings-guidance-2026-08-13/), [MarketBeat FY guidance](https://www.marketbeat.com/instant-alerts/audioeye-nasdaqaeye-issues-fy-2026-earnings-guidance-2026-08-13/), [Benzinga forecasts](https://www.benzinga.com/quote/AEYE/earnings-forecasts), [StockAnalysis](https://stockanalysis.com/stocks/AEYE). Consensus as of approximately Aug–Sep 2026 per these pages; refresh from FactSet or Zacks before use.

## 2. Prior Quarter (Q2 2026, reported Aug 13, 2026)

- Revenue ~$10.7M, +9% YoY; 42nd consecutive quarter of sequential revenue growth
- ARR $42.3M, +11% YoY, +$1.1M QoQ
- Adj. EBITDA ~$3.0M (28% margin) vs. $1.9M (20%) a year ago
- GAAP net loss $0.9M (-$0.07/share); prior-year quarter had a $1.4M contingent-consideration gain that did not recur
- Guidance: FY26 adj. EBITDA raised to ≥$12.7M (from ≥$12M); >$15M annualized run-rate adj. EBITDA exiting 2026
- CEO said management is evaluating excess-cash deployment, including **buybacks and dividends**

Sources: [SEC 8-K (Aug 13, 2026)](https://www.sec.gov/Archives/edgar/data/0001362190/000110465926095913/aeye-20260813xex99d1.htm), [Q2 call summary (Nasdaq)](https://www.nasdaq.com/articles/audioeye-q2-earnings-call-highlights), [Q2 call transcript (Benzinga)](https://www.benzinga.com/news/26/08/61196242/audioeye-q2-2026-earnings-call-transcript).

## 3. Key Metrics to Watch (ranked)

1. **Revenue vs. $10.95M guide midpoint and ~$11.1M+ consensus.** Hitting the Street number requires a beat on the guide. The guide is the buy-side anchor.
2. **ARR and net new ARR.** Q2 added $1.1M. A sequential slowdown would hurt the growth narrative more than the quarter's revenue.
3. **Adj. EBITDA vs. $3.4–3.6M guide**, plus margin trend toward the >$15M run-rate target.
4. **FY26 guide:** the revenue range ($43.5–44.0M) and whether the ≥$0.98 EPS / ≥$12.7M EBITDA floors are reaffirmed or raised.
5. **Capital allocation:** any buyback authorization or dividend announcement would be a new catalyst.
6. **GAAP EPS** (consensus is unclear, $0.23 vs. $0.26 adjusted). Check which basis the Street uses.

## 4. Scenarios

| Scenario | Revenue | EPS | Key Driver | Stock Reaction |
|---|---|---|---|---|
| Bull | ≥ $11.1M, ARR growth ≥ 11% | Beat | Revenue above consensus, FY EBITDA raised again, buyback/dividend announced | Upside; size not verified |
| Base | $10.9–11.1M, in guide | In line | Reaffirm FY guide, modest ARR growth | Muted move |
| Bear | < $10.85M (below guide) | Miss | ARR growth slows, guide cut or FY revenue trimmed | Downside |

- **Bull:** Management raises FY revenue, reports ARR growth above 11%, and announces capital returns.
- **Base:** Results land inside guidance and the FY outlook is reaffirmed.
- **Bear:** Revenue falls below the guide floor, or ARR growth stalls while the company still spends on growth.

**Historical reactions:** I could not find a reliable record of next-day moves. One low-reliability source says shares rose modestly after the Q4 2025 report despite an EPS miss. Pull the last 8 quarters of gap and next-day returns from Yahoo Finance or Benzinga before sizing.

## 5. Catalyst Checklist

1. **Revenue vs. $10.95M guide / ~$11.1M consensus.** A print at the bottom of the guide signals a miss versus the Street even if it is "in guidance."
2. **FY26 revenue guide ($43.5–44.0M).** Does the company hold the range or narrow it to the low end?
3. **ARR of $42.3M and its trajectory.** Watch for net new ARR at or above Q2's $1.1M.
4. **Capital return.** A buyback or dividend would be a clear narrative shift, since the CEO flagged excess cash.
5. **Q4/FY27 commentary.** Look for early color on the >$15M run-rate EBITDA target and any M&A.

## 6. Trading Setup

- **Recent stock performance:** not found in my searches. Check the current price, 30-day move, and 52-week range before the print.
- **Options-implied move:** not found. Pull the ATM straddle for the front-month expiry that covers the print.
- **Positioning note:** The Q3 revenue guide sits below the consensus range, so a "beat" on the Street number is less likely than a beat on the guide. The EPS and EBITDA guides look above Street, which gives the bull case a second lever.

## Gaps and Next Steps

- Confirm the earnings date and time (Oct 26 vs. Nov 3) with the company's IR page or an 8-K notice.
- Reconcile the EPS consensus ($0.23 vs. $0.26) with one data provider.
- Pull the implied move and 8-quarter earnings-day reaction history.
- I used 3 of 4 allowed searches and 0 of 3 page fetches. I did not open the Q2 release or the transcript in full, so segment-level detail is not covered.